**The problem:** `1 tbsp Beef` and `1 tbsp Mint Leaves` are **not** the same 15 g.  
- 1 tbsp fresh mint ~ 2.5 g (light, airy herb)  
- 1 tbsp raw minced beef ~ 15 g (dense muscle tissue)  
- 1 tbsp ground cumin ~ 7 g (fine powder, partially air-filled)

**Multi-entry disambiguation:** When USDA has multiple entries for the same food (e.g. beef has 200 portion rows for steaks/roasts), we:
1. Filter to entries matching the **needed unit** (tablespoon/teaspoon/cup/medium/each)
2. Among those, prefer **plain raw whole-food** descriptions (no brand, no cooking modifier) 
3. Take the **median** gram-weight/amount to reduce outlier influence
4. Record which USDA description was selected

Ingredients where USDA Foundation Foods has the target unit -> **Tier 1**  
Ingredients where USDA has no matching unit (e.g. spices, garlic, raw meats in tbsp) -> **Tier 2**


In [ ]:
import pandas as pd, numpy as np, re

#  4.1  Load SR Legacy (modifier column encodes the actual unit)
SR_DIR = 'FoodData_Central_sr_legacy_food_csv_2018-04'
sr_food = pd.read_csv(f'{SR_DIR}/food.csv')
sr_fp   = pd.read_csv(f'{SR_DIR}/food_portion.csv')
sr_fp   = sr_fp.merge(sr_food[['fdc_id', 'description']], on='fdc_id', how='left')

_UNIT_RE = [
    ('tsp',    r'\btsp\b|teaspoon'),
    ('tbsp',   r'\btbsp\b|tablespoon'),
    ('cup',    r'\bcup\b'),
    ('clove',  r'\bcloves?\b'),
    ('medium', r'\bmedium\b'),
    ('large',  r'\blarge\b'),
    ('small',  r'\bsmall\b'),
]

def _sr_unit(modifier) -> str:
    if pd.isna(modifier): return None
    m = modifier.lower()
    for unit, pat in _UNIT_RE:
        if re.search(pat, m): return unit
    return None

sr_fp['unit_type']  = sr_fp['modifier'].apply(_sr_unit)
sr_fp['desc_norm']  = sr_fp['description'].str.lower()
sr_fp['g_per_unit'] = sr_fp['gram_weight']
sr_fp = sr_fp[sr_fp['unit_type'].notna() & sr_fp['g_per_unit'].notna()]

print(f"SR Legacy: {len(sr_fp):,} portion rows with parsed units")


SR Legacy: 4,446 portion rows with parsed units


In [11]:
#  0.  Load recipe ingredients + CO₂e lookup
ingr_df = pd.read_csv('recipe_ingredients.csv')
ingr_df['ingredient_name'] = ingr_df['ingredient_name'].str.strip()
print(f"recipe_ingredients.csv : {len(ingr_df):,} rows, "
      f"{ingr_df['ingredient_name'].nunique()} unique ingredients")

match_df = pd.read_csv('ingredient_co2.csv')
n_unique = ingr_df['ingredient_name'].nunique()
print(f"ingredient_co2.csv     : {len(match_df)} / {n_unique} ingredients "
      f"({len(match_df)/n_unique*100:.0f}% CO₂e coverage)")


recipe_ingredients.csv : 103,908 rows, 58 unique ingredients
ingredient_co2.csv     : 58 / 58 ingredients (100% CO₂e coverage)


In [ ]:
# 4.3  Plainness scoring: prefer raw/simple over cooked/branded
def _plainness(desc: str) -> int:
    d = desc.lower()
    s = 0
    if re.search(r'\b(raw|fresh)\b', d):                                        s -= 3
    if re.search(r'\b(cooked|fried|roasted|baked|boiled|grilled|canned)\b', d): s += 3
    if re.search(r'\b(salted|seasoned|flavored|sweetened)\b', d):               s += 2
    if re.search(r'[A-Z]{3,}', desc[:40]):                                      s += 1  # ALL-CAPS brand
    return s

#  4.4  Synonym map: recipe ingredient → USDA search keyword 
# Only for cases where the ingredient name would not match USDA descriptions.
# None = no USDA entry → fall through to generic.
USDA_SYNONYMS = {
    'ghee'            : 'butter, salted',   # clarified butter
    'garam masala'    : 'curry powder',     # blend proxy
    'fresh coriander' : 'coriander leaf',
    'mint leaves'     : 'spearmint',
    'lentils (dal)'   : 'lentils',
    'mustard seeds'   : 'mustard seed',
    'sesame seeds'    : 'sesame seed',
    'tamarind paste'  : 'tamarind',
    'green chili'     : 'peppers, chili',
    'bell pepper'     : 'peppers, sweet',
    'paneer'          : 'cottage cheese',   # no USDA entry
    'fenugreek leaves': 'fenugreek',
    'basmati rice'    : 'rice, white',
    'curry leaves'    : None,               # absent from USDA; fall through to generic
    'carrot'          : 'carrots, raw', 
}


In [ ]:
# NOTE: Only SR Legacy is used for USDA lookups. Foundation Foods (2026 release)
# was evaluated but omitted: SR Legacy has broader coverage of culinary units
# (tablespoon, teaspoon, cup, medium) in its modifier column, while Foundation
# Foods portion data uses measure_unit_id mappings that require an additional
# join and cover fewer of the units present in this recipe dataset.
# SR Legacy is sufficient for the 58 ingredients in scope.

#  4.5  USDA lookup 
def _usda_g(keyword: str, unit: str, df: pd.DataFrame):
    """Keyword search → unit filter → median g/unit of 5 plainest matches."""
    mask = df['desc_norm'].str.contains(keyword.lower(), na=False, regex=False)
    sub  = df[mask & (df['unit_type'] == unit)].copy()
    if sub.empty: return None
    sub['_rank'] = sub['desc_norm'].apply(_plainness)
    vals = sub.nsmallest(5, '_rank')['g_per_unit'].dropna()
    return float(vals.median()) if not vals.empty else None

# Generic fallback - used only when USDA has no matching entry for the unit
_FALLBACK = {
    'tbsp': 15.0, 'tsp': 5.0, 'cup': 240.0, 'ml': 1.0,
    'g': 1.0, 'kg': 1000.0, 'clove': 4.0,
    'medium': 100.0, 'count_medium': 100.0,
    'large': 150.0, 'small': 60.0,
    'to_taste': 5.0, 'other': 100.0,
    'egg': 50.0, 'fillet': 150.0,
}

# Density map for ml -> grams conversion (g/ml)
# Used for liquid dairy/fat ingredients given in ml.
_DENSITY = {
    'milk'  : 1.03,
    'cream' : 1.01,
    'ghee'  : 0.90,
    'butter': 0.91,
    'yogurt': 1.03,
    'oil'   : 0.92,
    'water' : 1.00,
}

# Count-based ingredients: when unit='other' treat as a meaningful count unit
_COUNT_REMAP = {
    'egg'          : 'egg',
    'eggs'         : 'egg',
    'fish'         : 'fillet',
    'lentils'      : 'medium',
    'lentils (dal)': 'medium',
}

def get_grams(ingredient: str, unit: str) -> tuple:
    """Returns (g_per_unit, tier) where tier in {'sr_legacy', 'generic'}."""
    ingr_l  = ingredient.lower().strip()
    ingr_cl = re.sub(r'\s*\(.*?\)', '', ingr_l).strip()   # strip parentheticals
    lookup_unit = 'medium' if unit == 'count_medium' else unit

    # FIX: remap count-based 'other' units to meaningful unit for USDA lookup
    if lookup_unit == 'other' and ingr_l in _COUNT_REMAP:
        lookup_unit = _COUNT_REMAP[ingr_l]

    # FIX: ml → grams via density for liquid dairy/fat; skip USDA (no ml unit in DB)
    if lookup_unit == 'ml':
        for key, density in _DENSITY.items():
            if key in ingr_l:
                return density, 'generic'
        return 1.0, 'generic'  # default: water density

    # g is already in grams - passthrough (g_per_unit = 1.0)
    if lookup_unit == 'g':
        return 1.0, 'generic'

    if ingr_l in USDA_SYNONYMS:
        kw = USDA_SYNONYMS[ingr_l]
        keywords = [kw] if kw is not None else []
    else:
        keywords = list(dict.fromkeys(filter(None, [ingr_cl, ingr_l if ingr_l != ingr_cl else None])))

    for kw in keywords:
        g = _usda_g(kw, lookup_unit, sr_fp)
        if g is not None: return g, 'sr_legacy'

    return _FALLBACK.get(unit, _FALLBACK.get(lookup_unit, 100.0)), 'generic'

#  4.6  Parse quantities ─
def _parse_qty(qty_str: str) -> tuple:
    q = str(qty_str).lower().strip()
    if any(x in q for x in ['taste', 'needed', 'optional']): return 1.0, 'to_taste'
    m = re.match(r'^([\d.]+)', q)
    n = float(m.group(1)) if m else 1.0
    if   re.search(r'\bkg\b', q):              return n, 'kg'
    elif re.search(r'\begg\b', q):             return n, 'egg'           # before g check
    elif re.search(r'g$|\bgrams?\b', q):       return n, 'g'             # FIX: g$ catches "933g"
    elif re.search(r'\bml\b', q):              return n, 'ml'
    elif re.search(r'tbsp|tablespoon', q):     return n, 'tbsp'
    elif re.search(r'tsp|teaspoon', q):        return n, 'tsp'
    elif re.search(r'\bcup\b', q):             return n, 'cup'
    elif re.search(r'cloves?', q):             return n, 'clove'
    elif re.search(r'fillet', q):              return n, 'fillet'
    elif re.search(r'count_medium|medium', q): return n, 'count_medium'
    elif re.search(r'\blarge\b', q):           return n, 'large'
    elif re.search(r'\bsmall\b', q):           return n, 'small'
    else:                                      return n, 'other'


In [14]:
ingr_df2 = ingr_df.copy()
ingr_df2['ingredient_name'] = ingr_df2['ingredient_name'].str.strip()

# Parse quantities if not already present in the CSV.
# _parse_qty is defined in cell 5; this branch runs on a fresh recipe_ingredients.csv.
if 'qty_unit' not in ingr_df2.columns:
    qty_col = 'quantity' if 'quantity' in ingr_df2.columns else ingr_df2.columns[-1]
    parsed  = ingr_df2[qty_col].apply(
        lambda x: pd.Series(_parse_qty(x), index=['qty_amount', 'qty_unit'])
    )
    ingr_df2 = pd.concat([ingr_df2, parsed], axis=1)

# Apply USDA gram lookup (cached by unique ingredient × unit key)
_cache: dict = {}
def _cached(ingr: str, unit: str) -> tuple:
    k = (ingr.lower().strip(), unit)
    if k not in _cache:
        _cache[k] = get_grams(ingr, unit)
    return _cache[k]

res = ingr_df2.apply(
    lambda r: pd.Series(_cached(r['ingredient_name'], r['qty_unit']),
                        index=['g_per_unit', 'usda_tier']),
    axis=1
)
ingr_df2 = pd.concat([ingr_df2, res], axis=1)
ingr_df2['grams'] = ingr_df2['qty_amount'] * ingr_df2['g_per_unit']

print(f"Processed {len(ingr_df2):,} rows  |  "
      f"{ingr_df2['recipe_id'].nunique()} recipes  |  "
      f"{ingr_df2['ingredient_name'].nunique()} unique ingredients")


Processed 103,908 rows  |  9997 recipes  |  58 unique ingredients


In [15]:
#  4.7  Tier summary ─
n_rows = len(ingr_df2)
print(f"=== USDA source tier ===")
tier_counts = ingr_df2['usda_tier'].value_counts()
for tier, cnt in tier_counts.items():
    print(f"  {tier:<20} {cnt:>7,}  ({cnt/n_rows*100:.1f}%)")

#  4.8  Grounded g/unit table 
unit_table = (
    ingr_df2.groupby(['ingredient_name', 'qty_unit'])
    .agg(g_per_unit=('g_per_unit', 'first'),
         tier=('usda_tier', 'first'),
         n_rows=('grams', 'count'))
    .reset_index()
    .sort_values(['ingredient_name', 'qty_unit'])
)
print(f"\n=== Grounded g/unit table ({len(unit_table)} ingredient × unit combos) ===")
pd.set_option('display.max_colwidth', 55)
display(unit_table.reset_index(drop=True))


=== USDA source tier ===
  sr_legacy             54,981  (52.9%)
  generic               48,927  (47.1%)

=== Grounded g/unit table (71 ingredient × unit combos) ===


,ingredient_name,qty_unit,g_per_unit,tier,n_rows
0,All-Purpose Flour,g,1.00,generic,1608
1,Almonds,tbsp,9.10,sr_legacy,1116
2,Basmati Rice,g,1.00,generic,1549
3,Bay Leaf,tsp,0.60,sr_legacy,814
4,Beef,g,1.00,generic,892
...,...,...,...,...,...
66,Turmeric,tsp,3.00,sr_legacy,860
67,Vinegar,tbsp,14.90,sr_legacy,1350
68,Water,tbsp,15.00,sr_legacy,1353
69,Wheat Flour,g,1.00,generic,1588


In [16]:
#  4.9  Recompute recipe-level CO₂e ─
# Use lowercase map so title-cased ingredient_name matches lowercase match_df index
co2_lu = match_df.set_index('ingredient')['co2_kg_per_kg']
ingr_df2['co2_kg_per_kg'] = ingr_df2['ingredient_name'].str.lower().map(co2_lu)

# Diagnostic: list any ingredients missing from CO2e lookup
unmatched = (
    ingr_df2.loc[ingr_df2["co2_kg_per_kg"].isna(), "ingredient_name"]
    .str.lower().unique()
)
if len(unmatched):
    print(f"WARNING: {len(unmatched)} ingredient(s) missing CO2e: {sorted(unmatched)}")
else:
    print("All ingredients matched in CO2e lookup.")


# co2e_kg = grams × (kg_CO2e / kg_food) / 1000  →  result is kg CO2e per row
ingr_df2['co2e_kg'] = ingr_df2.apply(
    lambda r: r['grams'] * (r['co2_kg_per_kg'] / 1000)
              if pd.notna(r['co2_kg_per_kg']) else 0.0, axis=1
)

recipe_co2 = (
    ingr_df2.groupby('recipe_id')
    .agg(
        co2e_kg =('co2e_kg',        lambda x: x.sum()),  # already in kg CO2e
        n_ingr  =('ingredient_name','count'),
        n_sr    =('usda_tier',      lambda x: (x == 'sr_legacy').sum()),
        n_gen   =('usda_tier',      lambda x: (x == 'generic').sum()),
    )
    .reset_index()
)
recipe_co2['usda_pct']    = (recipe_co2['n_sr'] / recipe_co2['n_ingr'] * 100).round(1)
recipe_co2['generic_pct'] = (recipe_co2['n_gen'] / recipe_co2['n_ingr'] * 100).round(1)

n_matched = ingr_df2['co2_kg_per_kg'].notna().sum()
print(f"=== Recipe CO₂e (USDA-grounded) ===")
print(f"  Rows with CO₂e data     : {n_matched:,} / {len(ingr_df2):,} "
      f"({n_matched/len(ingr_df2)*100:.1f}%)")
print(f"  Recipes analysed        : {len(recipe_co2)}")
print(f"  Median CO₂e / recipe    : {recipe_co2['co2e_kg'].median():.3f} kg CO₂eq")
print(f"  Mean CO₂e / recipe      : {recipe_co2['co2e_kg'].mean():.3f} kg CO₂eq")
print(f"  Avg SR Legacy coverage  : {recipe_co2['usda_pct'].mean():.1f}%")
print(f"  Avg generic fallback    : {recipe_co2['generic_pct'].mean():.1f}%")

print(f"\nTop 10 highest-emission recipes:")
display(
    recipe_co2.nlargest(10, 'co2e_kg')
    [['recipe_id', 'co2e_kg', 'n_ingr', 'usda_pct', 'generic_pct']]
    .reset_index(drop=True)
)


All ingredients matched in CO2e lookup.
=== Recipe CO₂e (USDA-grounded) ===
  Rows with CO₂e data     : 103,908 / 103,908 (100.0%)
  Recipes analysed        : 9997
  Median CO₂e / recipe    : 9.041 kg CO₂eq
  Mean CO₂e / recipe      : 13.677 kg CO₂eq
  Avg SR Legacy coverage  : 52.6%
  Avg generic fallback    : 47.4%

Top 10 highest-emission recipes:


,recipe_id,co2e_kg,n_ingr,usda_pct,generic_pct
0,RCP02318,78.517507,14,50.0,50.0
1,RCP09663,72.050897,18,55.6,44.4
2,RCP02982,71.848890,13,53.8,46.2
3,RCP09711,70.465656,14,28.6,71.4
4,RCP00501,70.214479,12,50.0,50.0
5,RCP09759,69.839937,17,52.9,47.1
6,RCP02295,69.079830,9,44.4,55.6
7,RCP03645,68.553892,15,46.7,53.3
8,RCP08459,68.280045,11,54.5,45.5
9,RCP03933,67.912800,12,58.3,41.7


In [ ]:
#  4.10  Persist results 
ingr_df2.to_csv('recipe_ingredients_grounded.csv', index=False)
recipe_co2.to_csv('recipe_co2.csv', index=False)
print(f"Saved: recipe_ingredients_grounded.csv  ({len(ingr_df2):,} rows)")
print(f"Saved: recipe_co2.csv  ({len(recipe_co2):,} rows)")


Saved: recipe_ingredients_grounded.csv  (103,908 rows)
Saved: recipe_co2.csv  (9,997 rows)
